# Final evaluation on the test set

The test set (20% of the patients, never used so far) is opened once. For each treatment: the risk model's AUC, the average effect, and whether the causal forest fitted on the training set orders the test patients by their response.

Each treatment has its own stratified split. With about 3,900 test patients the uplift is computed on four groups instead of ten.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from ml_med.modeling import (
    bootstrap_uplift,
    fit_causal_forest,
    make_model,
    predict_cate_cf,
    prepare_data,
    split_train_test,
    uplift_by_decile,
)

df = pd.read_parquet(Path("../data/processed/ist_features.parquet"))


def ate_with_se(y, t):
    """Average treatment effect and its standard error."""
    y1, y0 = y[t == 1], y[t == 0]
    ate = y1.mean() - y0.mean()
    se = np.sqrt(y1.var() / len(y1) + y0.var() / len(y0))
    return ate, se

In [2]:
for treatment in ("RXASP", "RXHEP"):
    X, y, t = prepare_data(df, treatment)
    X_train, X_test, y_train, y_test, t_train, t_test = split_train_test(X, y, t)

    risk_model = make_model(treatment).fit(X_train, y_train)
    auc = roc_auc_score(y_test, risk_model.predict_proba(X_test)[:, 1])
    ate, se = ate_with_se(y_test, t_test)

    forest = fit_causal_forest(X_train, y_train, t_train, treatment)
    cate = pd.Series(predict_cate_cf(forest, X_test), index=X_test.index)
    table = uplift_by_decile(cate, y_test, t_test, n_bins=4)
    boot = bootstrap_uplift(cate, y_test, t_test, n_bins=4)
    diff = boot[[1, 2]].mean(axis=1) - boot[[3, 4]].mean(axis=1)

    print(f"===== {treatment}: {len(X_test)} test patients")
    print(f"risk model AUC: {auc:.3f}")
    print(f"ATE: {ate:+.4f} (standard error {se:.4f})")
    print(table.round(3).to_string())
    print(
        "groups 1-2 minus groups 3-4:",
        round(diff.mean(), 3),
        "95% interval:",
        diff.quantile([0.025, 0.975]).round(3).tolist(),
    )
    print()

===== RXASP: 3857 test patients
risk model AUC: 0.792
ATE: -0.0126 (standard error 0.0156)
          n  cate_mean  rate_control  rate_treated  uplift
decile                                                    
1       965     -0.029         0.410         0.452   0.042
2       964     -0.017         0.613         0.563  -0.050
3       964     -0.008         0.705         0.669  -0.036
4       964      0.001         0.804         0.814   0.009
groups 1-2 minus groups 3-4: 0.007 95% interval: [-0.052, 0.07]

===== RXHEP: 3857 test patients
risk model AUC: 0.811
ATE: +0.0002 (standard error 0.0156)
          n  cate_mean  rate_control  rate_treated  uplift
decile                                                    
1       965     -0.015         0.559         0.568   0.009
2       964     -0.005         0.607         0.615   0.007
3       964      0.002         0.663         0.637  -0.026
4       964      0.011         0.688         0.694   0.006
groups 1-2 minus groups 3-4: 0.019 95% interv

## Conclusion

1. The risk model holds on unseen patients. Test AUC is 0.79 (aspirin split) and 0.81 (heparin split), against 0.80 in cross-validation on the training set.

2. The average effects are the same as on the training set: −0.013 for aspirin and 0.000 for heparin. With a standard error of 0.016 the test set alone cannot tell the aspirin effect from zero; on all patients the standard error is about 0.007.

3. The causal forest's ordering does not replicate. For aspirin, the group with the largest predicted benefit shows an uplift of +0.042, and groups 1–2 minus groups 3–4 is +0.007 (95% interval −0.052 to +0.070): the hint from the training set, that milder strokes benefit more, is not confirmed. For heparin the same difference is +0.019 (−0.043 to +0.084).

4. Final verdict. The baseline risk of death or dependency can be predicted well. The average effect of each treatment can be estimated. The data do not support personalising the treatment effect.